In [1]:
from hexgrid import HexCoord, hdist, euclid_center_distance
from unet.hexunet_model import HexUNet
from hexplot import plot_hex_grid_2

import matplotlib.pyplot as plt

In [2]:
import random
import torch


def make_distance_sample(
    radius,
    center,
    seed,
    hex_size=1.0,
):
    rng = random.Random(seed)

    size = 2 * radius + 1

    x = torch.zeros(
        1,
        size,
        size,
        dtype=torch.float32,
    )

    y = torch.zeros(
        1,
        size,
        size,
        dtype=torch.float32,
    )

    valid_mask = torch.zeros(
        1,
        size,
        size,
        dtype=torch.bool,
    )

    while True:
        q = rng.randint(-radius, radius)
        r = rng.randint(-radius, radius)

        goal = HexCoord(
            center.q + q,
            center.r + r,
        )

        if hdist(center, goal) <= radius:
            break

    for q in range(-radius, radius + 1):
        for r in range(-radius, radius + 1):

            cell = HexCoord(
                center.q + q,
                center.r + r,
            )

            if hdist(center, cell) > radius:
                continue

            row = r + radius
            col = q + radius

            valid_mask[0, row, col] = True

            y[0, row, col] = euclid_center_distance(
                cell,
                goal,
                hex_size,
            )

    goal_row = goal.r - center.r + radius
    goal_col = goal.q - center.q + radius

    x[0, goal_row, goal_col] = 1.0

    return x, y, valid_mask, goal

In [3]:
x, y, valid_mask, goal = make_distance_sample(
    radius=15,
    center=HexCoord(0, 0),
    seed=1,
    hex_size=1.0,
)

print("Input:", x.shape)
print("Target:", y.shape)
print("Goal:", goal)
print("Goal target:", y[x.bool()])

Input: torch.Size([1, 31, 31])
Target: torch.Size([1, 31, 31])
Goal: HexCoord(q=-11, r=3)
Goal target: tensor([0.])


In [4]:
def make_distance_batch(
    batch_size,
    radius,
    center,
    seed,
    hex_size=1.0,
):
    inputs = []
    targets = []
    masks = []

    for i in range(batch_size):
        x, y, mask, _ = make_distance_sample(
            radius=radius,
            center=center,
            seed=seed + i,
            hex_size=hex_size,
        )

        inputs.append(x)
        targets.append(y)
        masks.append(mask)

    return (
        torch.stack(inputs),
        torch.stack(targets),
        torch.stack(masks),
    )

In [5]:
x, y, mask = make_distance_batch(
    batch_size=16,
    radius=15,
    center=HexCoord(0, 0),
    seed=0,
)

print(x.shape)
print(y.shape)
print(mask.shape)

torch.Size([16, 1, 31, 31])
torch.Size([16, 1, 31, 31])
torch.Size([16, 1, 31, 31])


In [6]:
def masked_mse_loss(
    prediction,
    target,
    mask,
):
    error = (
        prediction[mask]
        - target[mask]
    )

    return torch.mean(
        error ** 2
    )

In [ ]:
model = HexUNet(
    n_channels=1,
    n_classes=1,
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

for step in range(1000):

    x, y, mask = make_distance_batch(
        batch_size=16,
        radius=15,
        center=HexCoord(0, 0),
        seed=step * 16,
    )

    prediction = model(x)

    loss = masked_mse_loss(
        prediction,
        y,
        mask,
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())

0 629.6901245117188


In [ ]:
def tensor_to_hex_values(
    tensor,
    valid_mask,
    center,
    radius,
):
    values = {}

    tensor = tensor.detach().cpu()
    valid_mask = valid_mask.detach().cpu()

    for row in range(tensor.shape[-2]):
        for col in range(tensor.shape[-1]):

            if not valid_mask[0, row, col]:
                continue

            q = center.q + col - radius
            r = center.r + row - radius

            values[HexCoord(q, r)] = float(
                tensor[0, row, col]
            )

    return values

In [ ]:
def plot_hexunet_prediction(
    input_tensor,
    target_tensor,
    prediction_tensor,
    valid_mask,
    center,
    radius,
    hex_size=1.0,
    show_error=True,
):
    goal_index = torch.nonzero(
        input_tensor[0] > 0,
        as_tuple=False,
    )[0]

    goal_row = int(goal_index[0])
    goal_col = int(goal_index[1])

    goal = HexCoord(
        center.q + goal_col - radius,
        center.r + goal_row - radius,
    )

    target_values = tensor_to_hex_values(
        target_tensor,
        valid_mask,
        center,
        radius,
    )

    prediction_values = tensor_to_hex_values(
        prediction_tensor,
        valid_mask,
        center,
        radius,
    )

    error_values = {
        cell: abs(
            prediction_values[cell]
            - target_values[cell]
        )
        for cell in target_values
    }

    n_plots = 3 if show_error else 2

    fig, axes = plt.subplots(
        1,
        n_plots,
        figsize=(8 * n_plots, 8),
    )

    plot_hex_grid_2(
        obstacles=[],
        start=goal,
        goal=goal,
        center=center,
        radius=radius,
        hex_size=hex_size,
        ax=axes[0],
        path=target_values,
        plt_title="Target Distance",
        draw_axes=False,
        show_axis_legend=False,
    )

    plot_hex_grid_2(
        obstacles=[],
        start=goal,
        goal=goal,
        center=center,
        radius=radius,
        hex_size=hex_size,
        ax=axes[1],
        path=prediction_values,
        plt_title="Predicted Distance",
        draw_axes=False,
        show_axis_legend=False,
    )

    if show_error:
        plot_hex_grid_2(
            obstacles=[],
            start=goal,
            goal=goal,
            center=center,
            radius=radius,
            hex_size=hex_size,
            ax=axes[2],
            path=error_values,
            plt_title="Absolute Prediction Error",
            draw_axes=False,
            show_axis_legend=False,
        )

    plt.tight_layout()
    plt.show()

In [ ]:
x, target, mask, goal = make_distance_sample(
    radius=15,
    center=HexCoord(0, 0),
    seed=1234,
    hex_size=1.0,
)

model.eval()

with torch.no_grad():
    prediction = model(
        x.unsqueeze(0)
    )[0]

plot_hexunet_prediction(
    input_tensor=x,
    target_tensor=target,
    prediction_tensor=prediction,
    valid_mask=mask,
    center=HexCoord(0, 0),
    radius=15,
)